# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/da4b0/FlyRank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I picked this lane because it has the clearest link between a data output and a real human decision: a reviewer has limited time, and a ranked list tells them which page to look at first. The starter pipeline already shows that a learned ranking can beat a fixed rule on this kind of problem. On the 30,000-row starter slice, with whole clients held out for validation, the rule-based baseline got Precision@50 = 0.240 and the random forest got 0.740 (from `outputs/model_report.md`). That gap is worth 7 weeks of work, as long as I stay honest that the starter label is only a proxy. The section 3 numbers below back this choice with the data itself.

I can still confirm or change this lane until the end of Week 4.

In [ ]:
LANE = 'Lane 2: Refresh / Content Opportunity Scoring (provisional)'
print(LANE)


## 2. The question: decision, action, cost of a wrong call

- **Search question:** Which existing pages should a content team review first for a refresh, because they still have real search demand but their search visibility has been moving down?
- **Unit of analysis:** one page (one `content_id`) at one decision point.
- **Output:** a ranked review queue. Each page gets a score, a suggested action (for example refresh, review CTR, or monitor) and reason codes a human can read.
- **Decision and action:** the content team has limited capacity (say 20 to 50 pages per cycle). Someone on that team opens the top of the queue, checks each page by hand, and decides whether to refresh, rewrite the title/meta, or leave it alone.
- **Cost of a wrong call:**
  - *False alarm* (a page in the top of the queue was fine): a reviewer wastes some time, and a good page might get edited without need. This cost is small and visible.
  - *Missed problem* (a page that really is losing visibility never reaches the top): the loss continues unnoticed for longer. This cost is larger but hidden, so I will track both precision@K and recall.
  - Because a human checks every recommendation before anything is changed, a false alarm is cheap. That is why I lean towards catching more real problems, while still keeping the list short enough to review.
- **Why data or ML can help at all:** the inventory is too large to check page by page. Several weak signals (impressions, position, age, CTR, engagement) point in slightly different directions, and a person cannot weigh them across thousands of pages. Data can also separate real movement from small, low-volume wiggles.
- **Why this is not just "train a model":** the first step is a transparent rule-based score that I can explain line by line. A model only earns its place if it beats that baseline on a validation design that matches the real decision (client holdout, then a future time window). The deliverable is a ranked list with reasons and caveats for a human reviewer, not a prediction score on its own.

In [ ]:
FRAMING = {
    'unit': 'one page (content_id)',
    'output': 'ranked review queue with action + reason codes',
    'reviewer_capacity_assumed': '20-50 pages per cycle',
    'baseline_first': 'transparent weighted rule score',
    'metrics_that_match_the_decision': ['precision@K', 'recall', 'average precision'],
}
for k, v in FRAMING.items():
    print(f'{k}: {v}')


## 3. Quick look at the data (2-3 real numbers)

The cell below loads the starter CSV, applies the same starter filters as `scripts/01_prepare_features.py` (impressions above zero, page at least 90 days old, one row per page), and prints three numbers:

1. **Size of the review problem:** how many pages remain after the filters, what share carries the starter `down` label, and how small a reviewer's capacity of 50 pages is next to that.
2. **Pages that matter:** how many pages are both declining and have real demand (at least 100 impressions in 90 days), and how many of the visible pages (500+ impressions) are declining.
3. **Client spread:** how much the declining share differs between clients. A wide spread is the reason to validate with whole clients held out.

The starter label (`trend_direction == 'down'`) is computed from the current window, so it is a proxy and not a future outcome. These numbers show the problem is worth working on; they do not prove any model works.

In [ ]:
import os
import pandas as pd

CANDIDATES = [
    '../../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv',
    'https://raw.githubusercontent.com/da4b0/FlyRank/main/data/raw/content_refresh_anonymized.csv',
]
path = next(p for p in CANDIDATES if p.startswith('http') or os.path.exists(p))
df = pd.read_csv(path)
print('loaded', df.shape, 'from', 'GitHub raw URL' if path.startswith('http') else 'local file')

# Same starter filters as scripts/01_prepare_features.py
d = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id').copy()
d['is_declining'] = d['trend_direction'].eq('down')

# Number 1: size of the review problem
n_pages = len(d)
decl_share = d['is_declining'].mean()
print(f'1) pages after starter filters: {n_pages:,}')
print(f'   share labelled declining: {decl_share:.1%}')
print(f'   a reviewer checking 50 pages covers {50 / n_pages:.2%} of the inventory')

# Number 2: pages that matter (real demand)
declining_with_demand = d['is_declining'] & (d['impressions_90d'] >= 100)
visible = d['impressions_90d'] >= 500
visible_decl_share = d.loc[visible, 'is_declining'].mean()
print(f'2) declining with demand (>=100 impressions): {declining_with_demand.sum():,} pages ({declining_with_demand.mean():.1%} of all)')
print(f'   visible pages (>=500 impressions): {visible.sum():,}; share of them declining: {visible_decl_share:.1%}')

# Number 3: spread between clients
by_client = d.groupby('client_id')['is_declining'].agg(['mean', 'size'])
big = by_client[by_client['size'] >= 100]
print(f'3) clients with 100+ pages: {len(big)}')
print(f'   declining share per client: min {big["mean"].min():.1%}, median {big["mean"].median():.1%}, max {big["mean"].max():.1%}')


## 4. Careful words: what I can and can't claim

**What this work can say (observed, directional, decision-support):**
- In this sample, pages with certain signals (for example high impressions with a declining trend) are more likely to carry the declining label.
- A ranked list built from observable signals can help a reviewer decide which pages to check first, and I can measure whether it does that better than a simple rule on held-out clients.
- Results suggest where to look; every recommendation still needs a human check.

**What this work will not say:**
- That a refresh *caused* a recovery. This data has no experiment or causal design.
- That I know how Google ranks pages, or that a signal is a ranking factor.
- That results from the 30,000-row starter slice hold for the full warehouse, for other clients, or for other time periods.
- That a `down` trend bucket is a real, lasting decline. It is a proxy; consolidation, seasonality, SERP changes and low-volume noise can all look the same.

I will also avoid using any product score or flag as both feature and label, and keep all client names, URLs and raw queries out of the notebook.

In [ ]:
# No code needed for this section.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.